# Step 3: Learning

<!-- notebook-only -->

> **How to use this notebook**
> - **Just reading?** The text cells contain the whole lesson. You can skip every code cell. An even nicer way to read: the website, built with `.venv/bin/python website/build.py --serve`.
> - **Want to practice?** Run the code cells from top to bottom (Shift+Enter). They train the toddler for real and print every step. In **Your turn** cells, replace `None` with your answer: `check(...)` says if it is right and gives a hint if not, and `solution("3.1")` shows the worked answer.
> - Every question also has a **Show the answer** box you can open.
> - At the end, a **playground** trains the toddler on your own sentences.

<!-- /notebook-only -->

**Goal:** start from random numbers and let the toddler find its own numbers: guess, measure the surprise, push every number a little, and repeat.

Until now we wrote the numbers ourselves (our hint rows of Step 2). From this step on, the toddler learns them.

In [ ]:
# Run this first (practice only): imports and the tokenizer (Step 1)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "toddler").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from toddler.bpe import BPETokenizer
from toddler.data import CORPUS
from toddler.embeddings import hint_table, random_table
from toddler.exercises import check, solution

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.float_format", "{:.2f}".format)

tok = BPETokenizer.train(CORPUS)
V = len(tok.vocab)
print(V, "tokens in the vocabulary")

## 3.1 The smallest toddler that can guess

To learn, the toddler must make guesses. The smallest model that can guess has **two tables**, both with one row of 8 numbers per token:

- the **input table**: the embedding table of Step 2;
- the **output table**: a second table of the same shape.

To guess what comes after a token:

1. Read the token's row in the input table. This is its vector, called **x**.
2. Take the dot product of x with **every** row of the output table. This gives one **score** per token: 53 scores.
3. Turn the 53 scores into probabilities (section 3.2).

![The smallest toddler: one guess forward, the pushes backward](figures/03_smallest_model.png)

It only looks at the last token, like the counting model of Step 0. The difference: its numbers are **learned**, not counted. This recipe will grow into a transformer: the next steps only add parts between the two tables.

Why a second table? An output row answers "when should I come next?", which is a different question from "what am I?". Some models use the same table for both (GPT-2 does); many others keep two tables.

## 3.2 From scores to probabilities: softmax

We work by hand on a **mini example**: the guess after `_want`. To keep it small, pretend there are only 3 candidates (milk, mommy and bed) and that vectors have only 2 numbers. The right answer is `_milk` (sentence 1).

```
x = input row of _want   = ( 1,    1  )
output row of _milk      = ( 0.5, −0.5)
output row of _mommy     = ( 1,    0  )
output row of _bed       = (−0.5, −0.5)
```

**Scores** (dot products):
- milk: 1 × 0.5 + 1 × (−0.5) = **0**
- mommy: 1 × 1 + 1 × 0 = **1**
- bed: 1 × (−0.5) + 1 × (−0.5) = **−1**

Scores can be negative, and they don't add up to 1, so they are not probabilities yet. **Softmax** turns them into probabilities in two moves:

1. Raise e (2.718…) to the power of each score. The result is always positive, and a bigger score gives a much bigger number.
2. Divide each result by the total, so they add up to 1.

| | score | e^score | ÷ 4.086 = probability |
|---|---|---|---|
| milk (right answer) | 0 | 1 | **0.245** |
| mommy | 1 | 2.718 | 0.665 |
| bed | −1 | 0.368 | 0.090 |
| total | | 4.086 | 1 |

For now, the toddler thinks mommy is the most likely, and gives the right answer only 24.5%.

In [ ]:
# Try it: the mini example, scores and softmax
x = np.array([1.0, 1.0])                  # input row of _want
mini_output = np.array([[0.5, -0.5],      # output row of _milk
                        [1.0, 0.0],       # output row of _mommy
                        [-0.5, -0.5]])    # output row of _bed
names = ["milk", "mommy", "bed"]
right = 0                                 # the right answer is milk


def softmax(scores):
    exps = np.exp(scores - scores.max())  # subtracting the max avoids huge numbers and changes nothing
    return exps / exps.sum()


scores = mini_output @ x
probs = softmax(scores)
print("scores       :", scores)
print("e^score      :", np.exp(scores), "  total:", round(np.exp(scores).sum(), 3))
print("probabilities:", probs)

## 3.3 Measuring the surprise: the loss

The **loss** of a guess says how surprised the toddler is by the right answer:

**loss = −ln(probability it gave to the right answer)**

(ln is the natural logarithm: the `ln` key of a calculator.)

| probability of the right answer | 1 | 0.9 | 0.5 | 0.33 | 0.245 | 0.1 | 0.01 |
|---|---|---|---|---|---|---|---|
| loss | 0 | 0.11 | 0.69 | 1.10 | **1.41** | 2.30 | 4.61 |

- Sure and right (probability 1): no surprise, the loss is 0.
- The less probability the right answer got, the bigger the surprise. The loss is never below 0.

In the mini example, milk got 0.245, so the loss is −ln(0.245) = **1.41**.

Real LLMs use exactly this loss. Its official name is **cross-entropy**.

In [ ]:
# Try it: the loss of the mini example, and the loss for a few probabilities
loss = -np.log(probs[right])
print(f"loss = -ln({probs[right]:.3f}) = {loss:.2f}\n")
for p in [1, 0.9, 0.5, 1 / 3, 0.245, 0.1, 0.01]:
    print(f"probability {p:<6.3g} → loss {-np.log(p) + 0.0:.2f}")  # + 0.0 turns -0.0 into 0.0

## 3.4 The push: which way should each number move?

To lower the surprise, the numbers must change. But which way, and by how much? The **push** on a number answers one question:

> If this number grows by a tiny bit, how much does the surprise grow?

Mathematicians call it the **gradient**. Its sign gives the direction, and its size says how much this number matters.

### The push on each score

For softmax followed by this loss, the rule is always the same:

- the right answer's score: **probability − 1**
- every other score: **probability**

| | probability | push on its score |
|---|---|---|
| milk (right answer) | 0.245 | 0.245 − 1 = **−0.755** |
| mommy | 0.665 | **+0.665** |
| bed | 0.090 | **+0.090** |

How to read it:
- **Negative push** (milk): growing this score *lowers* the surprise, so milk's score should go up.
- **Positive push** (mommy, bed): growing these scores *raises* the surprise, so they should go down. Mommy took the most probability, so it gets the biggest push.

You don't have to trust the rule: test it. If the milk score grows by 0.001, the loss changes by −0.000755, which is −0.755 × 0.001. The cell below does this test for every score.

In [ ]:
# Try it: the push on each score, then test it by nudging each score by 0.001
push_scores = probs.copy()
push_scores[right] -= 1
print("push on the scores:", push_scores, "\n")

for j, name in enumerate(names):
    nudged = scores.copy()
    nudged[j] += 0.001
    change = -np.log(softmax(nudged)[right]) - loss
    print(f"{name:<5} score + 0.001 → the loss changes by {change:+.6f}   push × 0.001 = {push_scores[j] * 0.001:+.6f}")

### The push on the output rows

The milk score is x₁ × milk₁ + x₂ × milk₂, where milk₁ is the first number of milk's output row. If milk₁ grows by a tiny bit, the milk score grows by x₁ times that bit, and nothing else changes. So:

**push on an output row = (push on its score) × x**

With x = (1, 1):
- milk: −0.755 × (1, 1) = **(−0.755, −0.755)**
- mommy: 0.665 × (1, 1) = **(0.665, 0.665)**
- bed: 0.090 × (1, 1) = **(0.090, 0.090)**

### The push on the input row x

If x₁ grows by a tiny bit, **every** score grows: each one by its own row's first number times that bit. So we add up the effects on the three scores:

**push on x = the sum, over the tokens, of (push on its score) × (its output row)**

- from milk: −0.755 × (0.5, −0.5) = (−0.378, +0.378)
- from mommy: 0.665 × (1, 0) = (0.665, 0)
- from bed: 0.090 × (−0.5, −0.5) = (−0.045, −0.045)
- **total: (0.243, 0.333)**

(Adding the rounded numbers gives 0.242; with all the decimals, it is 0.2426.)

This is **backpropagation**: the push flows backwards, from the surprise to the scores, then from each score to every number that made it. In a real LLM it flows back through many more layers, always with these same moves.

In [ ]:
# Try it: the pushes on the output rows and on x
push_output = np.outer(push_scores, x)    # row j = (push on score j) × x
push_x = push_scores @ mini_output        # sum over j of (push on score j) × (output row j)
for name, push in zip(names, push_output):
    print(f"push on the output row of {name:<5}: {push}")
print("push on x (the input row of _want):", push_x)

## 3.5 One learning step

Every number moves a little **against** its push:

**new = old − learning rate × push**

The push points toward *more* surprise, so we step the other way. The **learning rate** sets the size of the step. Here it is 0.5.

| number | old | push | new = old − 0.5 × push |
|---|---|---|---|
| x (input row of `_want`) | (1, 1) | (0.243, 0.333) | (0.879, 0.834) |
| output row of milk | (0.5, −0.5) | (−0.755, −0.755) | (0.878, −0.122) |
| output row of mommy | (1, 0) | (0.665, 0.665) | (0.667, −0.333) |
| output row of bed | (−0.5, −0.5) | (0.090, 0.090) | (−0.545, −0.545) |

Now guess again with the new numbers:

| | new score | probability before | probability after |
|---|---|---|---|
| milk (right answer) | 0.67 | 24.5% | **52.7%** |
| mommy | 0.31 | 66.5% | 36.7% |
| bed | −0.93 | 9.0% | 10.6% |

![One learning step on the mini example](figures/03_one_step.png)

The loss falls from 1.41 to **0.64** in one step. **All four rows moved at once**, the input row and the output rows: nothing is trained first.

Bed went up a little even though its push said "down": mommy lost a lot of probability, and some of it went to bed. A push is only sure for a tiny step, when the other numbers barely move. That's why the learning rate must not be too big: then the numbers jump past the good values, and the loss goes up and down. If it's too small, learning is slow.

In [ ]:
# Try it: one learning step, then guess again
learning_rate = 0.5
new_x = x - learning_rate * push_x
new_output = mini_output - learning_rate * push_output
new_scores = new_output @ new_x
new_probs = softmax(new_scores)

print("x:", x, "→", new_x)
for name, old_row, new_row in zip(names, mini_output, new_output):
    print(f"output row of {name:<5}: {old_row} → {new_row}")
print()
for j, name in enumerate(names):
    print(f"{name:<5}  score {scores[j]:+.2f} → {new_scores[j]:+.2f}   probability {probs[j]:.1%} → {new_probs[j]:.1%}")
print(f"\nloss: {loss:.2f} → {-np.log(new_probs[right]):.2f}")

## 3.6 The real toddler, at birth

Now the real thing, with the same recipe:

- **Input table:** 53 rows × 8 numbers, random. It is exactly the table at "birth" of Step 2.
- **Output table:** 53 rows × 8 numbers, random too (made the same way, with another seed).
- **Training data:** our 3 sentences. Each one gives 7 guesses (as in Step 0), so there are **21 guesses** in total:

| guess | token seen | right answer |
|---|---|---|
| 1 | `_I` | `_am` |
| 2 | `_am` | `_hungry` |
| 3 | `_hungry` | `_.` |
| 4 | `_.` | `_I` |
| 5 | `_I` | `_want` |
| 6 | `_want` | `_milk` |
| 7 | `_milk` | `_.` |
| 8 to 21 | the same for the other 2 sentences | |

At birth, here is the toddler's guess after `_I`. It is worked out exactly like the mini example, but with 53 scores:

| token | `_mom` | `n` | `_mommy` | `w` | `b` | … | `_am` (right answer) |
|---|---|---|---|---|---|---|---|
| probability | 4.5% | 4.3% | 4.0% | 3.9% | 3.7% | … | 2.2% |

It babbles: the right answer gets only 2.2%, a loss of −ln(0.022) = 3.82. On average over the 21 guesses, the loss at birth is **4.13**.

Compare: giving each of the 53 tokens the same chance (1/53) would give a loss of −ln(1/53) = 3.97. At birth, the toddler is even a little worse than that.

In [ ]:
# Try it: the two tables at birth, the 21 guesses, and the first guess
input_table = random_table(V, 8, seed=0)     # the table at birth of Step 2
output_table = random_table(V, 8, seed=1)

guesses = []                                 # (token seen, its position, right answer), as IDs
for line in CORPUS:
    ids = tok.encode(line)
    for position in range(1, len(ids)):
        guesses.append((ids[position - 1], position, ids[position]))
print(len(guesses), "guesses. The first 7:", [(tok.vocab[seen], tok.vocab[answer]) for seen, _, answer in guesses[:7]])

birth_probs = softmax(output_table @ input_table[tok.token_to_id["_I"]])
best = np.argsort(-birth_probs)[:5]
print("\nat birth, after _I:", ", ".join(f"{tok.vocab[i]} {birth_probs[i]:.1%}" for i in best),
      f"... _am {birth_probs[tok.token_to_id['_am']]:.1%}")

## 3.7 Training: one round, then many

**One round** = the 21 guesses, then one step:

1. For each guess, compute the scores, the probabilities, the loss and the pushes, exactly as in the mini example.
2. **Add up** the pushes of all the guesses. A row used by several guesses collects all their pushes. For example, the input row of `_I` is used by 6 guesses (`_am` comes after it 3 times, and `_want` 3 times), and every output row gets a push from every guess.
3. Divide by 21 to get the **average** push, then take one step: new = old − 0.5 × average push.

Then do it again, and again.

The 43 unused tokens never appear in the text, so their **input** rows get no push and never move. Their **output** rows are pushed down in every guess, because they are never the right answer.

In [ ]:
# Try it: one round of training, written out
def average_loss(input_table, output_table, guesses):
    total = 0.0
    for token, position, answer in guesses:
        total += -np.log(softmax(output_table @ input_table[token])[answer])
    return total / len(guesses)


def train_round(input_table, output_table, guesses, learning_rate=0.5):
    push_input = np.zeros_like(input_table)
    push_output = np.zeros_like(output_table)
    total_loss = 0.0
    for token, position, answer in guesses:
        x = input_table[token]
        probs = softmax(output_table @ x)
        total_loss += -np.log(probs[answer])
        push_scores = probs.copy()
        push_scores[answer] -= 1
        push_output += np.outer(push_scores, x)
        push_input[token] += push_scores @ output_table
    input_table -= learning_rate * push_input / len(guesses)     # all the numbers move, in the same step
    output_table -= learning_rate * push_output / len(guesses)
    return total_loss / len(guesses)                             # the loss before the step


trained_input, trained_output = input_table.copy(), output_table.copy()
for round_number in range(1, 4):
    loss_before = train_round(trained_input, trained_output, guesses)
    print(f"round {round_number}: loss {loss_before:.3f} → {average_loss(trained_input, trained_output, guesses):.3f}")

## 3.8 Watch it learn: 300 rounds

The average loss over the 21 guesses, with a learning rate of 0.5:

| after round | 0 (birth) | 5 | 10 | 25 | 50 | 100 | 200 | 300 |
|---|---|---|---|---|---|---|---|---|
| loss | 4.13 | 3.40 | 2.69 | 0.95 | 0.59 | 0.53 | 0.52 | **0.516** |

![The loss falls, round after round, down to a floor](figures/03_loss_curve.png)

Slow at first, then very fast, then slow again as it gets close to a **floor** at 0.512. Section 3.10 explains where the floor comes from.

In [ ]:
# Try it: train for 300 rounds from the tables at birth, and draw the loss
trained_input, trained_output = input_table.copy(), output_table.copy()
losses = [train_round(trained_input, trained_output, guesses) for _ in range(300)]
losses.append(average_loss(trained_input, trained_output, guesses))    # losses[k] = loss after k rounds
for k in [0, 5, 10, 25, 50, 100, 200, 300]:
    print(f"after {k:>3} rounds: loss {losses[k]:.3f}")

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.axhline(0.512, linestyle="--", color="gray")
plt.xlabel("rounds of training")
plt.ylabel("average loss")
plt.show()

## 3.9 What it learned: the counts of Step 0

After 300 rounds, the toddler's guesses:

| after | the toddler says |
|---|---|
| `_I` | am 50%, want 50% |
| `_am` | hungry 33%, sad 33%, tired 33% |
| `_want` | milk 33%, mommy 33%, bed 33% |
| `_.` | I 100% |
| `_hungry`, `_sad`, `_tired` | . 100% |
| `_milk`, `_mommy`, `_bed` | . 100% |

These are **exactly the probabilities of the counting model** of Step 0. Nobody gave it the counts: it found them by guessing, measuring its surprise and pushing its numbers. The 43 unused tokens share less than 1%.

In [ ]:
# Try it: the trained toddler's top 3 guesses after each token of our sentences
def top_guesses(input_table, output_table, token, k=3):
    probs = softmax(output_table @ input_table[tok.token_to_id[token]])
    best = np.argsort(-probs)[:k]
    return ", ".join(f"{tok.vocab[i]} {probs[i]:.0%}" for i in best)


used = ["_I", "_am", "_want", "_.", "_hungry", "_sad", "_tired", "_milk", "_mommy", "_bed"]
for token in used:
    print(f"after {token:<8} → {top_guesses(trained_input, trained_output, token)}")

## 3.10 Why the loss stops at 0.512

The toddler only sees the last token, so some guesses can never be sure:

| guesses | how many | best possible probability | loss of each |
|---|---|---|---|
| after `_I` (am or want?) | 6 | 1/2 | −ln(1/2) = 0.69 |
| after `_am` (which feeling?) | 3 | 1/3 | −ln(1/3) = 1.10 |
| after `_want` (which thing?) | 3 | 1/3 | 1.10 |
| all the others | 9 | 1 | 0 |

Average: (6 × 0.69 + 3 × 1.10 + 3 × 1.10 + 9 × 0) / 21 = **0.512**.

This is the best this model can do, however long it trains. To go lower, it needs more information:

- **Positions** (Step 4) tell the first `_I` from the second: the floor drops to **0.314**.
- **Attention** (Step 5) lets `_want` look back at the feeling: the floor drops to **0.157**.
- The 3 guesses after `_am` stay at 1/3 forever: nobody can know the feeling in advance. That part of the floor stays, even for a perfect model.

In [ ]:
# Try it: the floors, computed
floor = (6 * -np.log(1 / 2) + 6 * -np.log(1 / 3)) / 21
print(f"floor of this model: {floor:.3f}   loss after 300 rounds: {losses[-1]:.3f}")
print(f"floor with positions (Step 4): {6 * -np.log(1 / 3) / 21:.3f}")
print(f"floor with attention (Step 5): {3 * -np.log(1 / 3) / 21:.3f}")

## 3.11 What happened to the rows

The row of `_hungry` in the input table:

```
at birth:          -0.32  0.37 -0.59 -0.72  0.32  0.38 -0.48  0.28
after 300 rounds:  -0.59 -0.17  0.01 -0.25  0.21  0.93 -0.79  1.67
```

Random-looking decimals, like in real models: nobody can read the columns. But we can compare **directions**. The **cosine similarity** of two vectors is their dot product after scaling both to length 1:

**cosine(a, b) = a · b ÷ (length of a × length of b)**, where the length of a is √(a · a).

It is 1 when two vectors point the same way, 0 when they have nothing in common, and −1 when they point in opposite directions.

![Cosine similarity of the rows: the input table at birth and after training, and the output table after training](figures/03_rows_before_after.png)

- **Input table, at birth:** random similarities, from −0.55 to 0.57 between the six words in the box.
- **Input table, after training:** hungry, sad, tired, milk, mommy and bed all point the same way (0.5 to 0.9). Why? They are all followed by `.`, so they must all give the same guess.
- **Output table, after training:** the feelings group together (they all come after `am`), the things group together (they all come after `want`), and so do `_am` and `_want` (both come after `I`).

So the **input** row of a token learns *what comes after it*, and its **output** row learns *what comes before it*. Training doesn't know our column names from Step 2: it arranges the numbers in whatever way helps it guess. Our hint said "hungry and milk share food". The toddler doesn't need that link yet, because it can't look back. It will need it once attention lets `_want` see `_hungry` (Step 5).

In [ ]:
# Try it: cosine similarity of the input rows after training
def cosine_table(table, tokens):
    vectors = np.array([table[tok.token_to_id[t]] for t in tokens])
    vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)    # every row now has length 1
    return pd.DataFrame(vectors @ vectors.T, index=tokens, columns=tokens)


print("_hungry at birth        :", input_table[tok.token_to_id["_hungry"]])
print("_hungry after 300 rounds:", trained_input[tok.token_to_id["_hungry"]])
cosine_table(trained_input, used)

In [ ]:
# Try it: the output rows after training (feelings together, things together, am with want)
cosine_table(trained_output, used)

## 3.12 Random start, or our hint?

The input table can also start from our hint rows of Step 2 instead of random numbers (the output table stays random):

| after round | 0 | 10 | 25 | 50 | 100 | 300 |
|---|---|---|---|---|---|---|
| random start | 4.13 | 2.69 | 0.95 | 0.59 | 0.53 | **0.516** |
| hint start | 3.79 | 2.36 | 0.89 | 0.58 | 0.53 | **0.516** |

The hint helps a little at the start, then both end in the same place. **The starting point barely matters: the data decides.**

Real models always start from small random numbers. Why not from zeros? Look at the pushes: the push on an output row is multiplied by x, and the push on x is made of output rows. If both tables are all zeros, every push is 0, and nothing ever moves.

In [ ]:
# Try it: start from our hint rows, then from all zeros
def train_from(start_input, start_output, rounds=300):
    tables = [start_input.copy(), start_output.copy()]
    losses = [train_round(*tables, guesses) for _ in range(rounds)]
    return losses + [average_loss(*tables, guesses)]


hint_losses = train_from(hint_table(tok), output_table)
for k in [0, 10, 25, 50, 100, 300]:
    print(f"after {k:>3} rounds: random start {losses[k]:.3f}   hint start {hint_losses[k]:.3f}")

zero_losses = train_from(np.zeros((V, 8)), np.zeros((V, 8)), rounds=50)
print(f"\nall zeros: loss {zero_losses[0]:.3f} at birth, {zero_losses[-1]:.3f} after 50 rounds (nothing moves)")

## 3.13 How real LLMs learn

The same loop, at a giant scale:

- **Guesses:** trillions of them. Each step uses a **batch** of millions of tokens (not the whole training data), so steps are quick, and each push is an average over many texts.
- **The step rule:** usually **Adam** instead of our plain "old − learning rate × push". Adam gives every number its own step size, based on its recent pushes. The learning rate also changes during training: it starts small, grows, then slowly shrinks.
- **The pushes:** computed automatically. Libraries like PyTorch record every operation of the guess, then run backpropagation for you (this is called **autograd**). Our two push rules are exactly what it computes for these two tables.
- **The loss:** exactly ours, −ln(probability of the right token), the cross-entropy. You may also hear about **perplexity**, which is e^loss: roughly, how many tokens the model hesitates between. For our trained toddler, e^0.516 ≈ 1.7.
- **The cost:** one training run takes weeks on thousands of GPUs.

## 3.14 What is still missing

The toddler now **learns**, but it only sees the last token:

- After `_I`, it says am 50%, want 50%: it can't tell the first `_I` from the second. That's **Step 4, positions**, and the toddler will train again with them.
- After `_want`, it says ⅓ each: it can't look back at `_hungry`. That's **Step 5, attention**.

From now on, each step adds a part between the two tables, trains the whole toddler again from random numbers, and measures how far the loss falls.

<!-- notebook-only -->

The same code is saved in `toddler/learning.py`, and the next notebooks import it from there. It also accepts positions (Step 4).

<!-- /notebook-only -->

In [ ]:
# Try it: the package version used by the next notebooks trains the same way
from toddler import learning

assert learning.make_guesses(tok, CORPUS) == guesses
package_input, package_output, package_losses = learning.train(input_table, output_table, guesses, rounds=300)
assert np.allclose(package_losses, losses) and np.allclose(package_input, trained_input)
print(f"toddler/learning.py trains the same way: loss {package_losses[0]:.3f} → {package_losses[-1]:.3f}")

<!-- notebook-only -->

## Playground: your own toddler world

Write your own training data in the cell below: **up to 10 lines**. Add sentences, invent new words, remove a line, or repeat one to make it more frequent. Then run the cells of this section and look at what changes:

- the tokenizer learns new pieces (Step 1),
- the toddler starts from random numbers and trains,
- its guesses become the counts of your lines,
- its input rows group the tokens that are followed by the same things.

Ideas to try:
- Add `"I am happy . I want ball ."`: after `_am`, each feeling drops to 25%.
- Add `"I am hungry . I want milk ."` a second time: after `_want`, milk rises to 50%.
- Set `START = "hint"` to start from our hint rows of Step 2, or change `ROUNDS`.

<!-- /notebook-only -->

In [ ]:
# Playground: your own toddler world (up to 10 lines), then run this cell and the next ones
MY_CORPUS = [
    "I am hungry . I want milk .",
    "I am sad . I want mommy .",
    "I am tired . I want bed .",
    "I am happy . I want ball .",
]
START = "random"   # "random", or "hint": our hint rows of Step 2 (for the words that have one)
ROUNDS = 300

In [ ]:
# Playground: train the toddler on MY_CORPUS
from toddler import learning
from toddler.data import check_corpus

my_corpus = check_corpus(MY_CORPUS)
my_tok = BPETokenizer.train(my_corpus)
my_guesses = learning.make_guesses(my_tok, my_corpus)
my_start = hint_table(my_tok) if START == "hint" else random_table(len(my_tok.vocab), 8, seed=0)
my_input, my_output, my_losses = learning.train(my_start, random_table(len(my_tok.vocab), 8, seed=1), my_guesses,
                                                rounds=ROUNDS)

print(f"{len(my_corpus)} lines → {len(my_tok.vocab)} tokens in the vocabulary, {len(my_guesses)} guesses")
print(f"loss: {my_losses[0]:.3f} at birth → {my_losses[-1]:.3f} after {ROUNDS} rounds")
plt.figure(figsize=(6, 2.5))
plt.plot(my_losses)
plt.xlabel("rounds of training")
plt.ylabel("average loss")
plt.show()

In [ ]:
# Playground: what the trained toddler says after each token, next to the counts of your lines
from collections import Counter, defaultdict

heard = defaultdict(Counter)                  # token seen → how often each token came next
for seen, _, answer in my_guesses:
    heard[seen][answer] += 1

report = {}
for seen, followers in heard.items():
    my_probs = learning.softmax(my_output @ my_input[seen])
    total = sum(followers.values())
    report[my_tok.vocab[seen]] = {
        "the trained toddler says": ", ".join(f"{my_tok.vocab[i]} {my_probs[i]:.0%}" for i in np.argsort(-my_probs)[:3]),
        "the counts say": ", ".join(f"{my_tok.vocab[i]} {n / total:.0%}" for i, n in followers.most_common(3)),
    }
pd.DataFrame(report).T

In [ ]:
# Playground: which input rows point the same way after training (cosine similarity, 1 = same direction)
seen_ids = list(heard)
seen_names = [my_tok.vocab[i] for i in seen_ids]
vectors = my_input[seen_ids] / np.linalg.norm(my_input[seen_ids], axis=1, keepdims=True)
pd.DataFrame(vectors @ vectors.T, index=seen_names, columns=seen_names)

## Your turn

Do each one **by hand first** (a calculator is fine).

### 3.1 Softmax by hand: the scores are milk 2, mommy 0 and bed 0. What are the 3 probabilities?

<details>
<summary>Show the answer</summary>
<!-- answer: 0.79 0.11 0.11 | 0.787 0.107 0.107 | 0.787 0.106 0.106 | 0.79 0.1 0.1 | 0.8 0.1 0.1 -->
<!-- format: three numbers, e.g. 0.5 0.25 0.25 -->

| | score | e^score | ÷ 9.389 = probability |
|---|---|---|---|
| milk | 2 | 7.389 | **0.787** |
| mommy | 0 | 1 | **0.107** |
| bed | 0 | 1 | **0.107** |
| total | | 9.389 | 1 |

A score only 2 points higher gets about 7 times more probability.

</details>

In [ ]:
my_answer = None  # three numbers, e.g. [0.5, 0.25, 0.25]
check("3.1", my_answer)

### 3.2 The toddler gave the right answer a probability of 0.25. What is the loss?

<details>
<summary>Show the answer</summary>
<!-- answer: 1.39 | 1.386 | 1.3863 | 1.4 -->
<!-- format: one number -->

loss = −ln(0.25) = **1.39**. It's the loss of a toddler hesitating between 4 equally likely answers.

</details>

In [ ]:
my_answer = None  # one number
check("3.2", my_answer)

### 3.3 The probabilities are milk 0.7, mommy 0.2 and bed 0.1, and the right answer is mommy. What is the push on each score?

<details>
<summary>Show the answer</summary>
<!-- answer: 0.7 -0.8 0.1 | milk 0.7 mommy -0.8 bed 0.1 -->
<!-- format: three numbers (milk, mommy, bed), e.g. 0.5 -0.5 0 -->

- milk: **0.7** (a wrong answer gets its probability)
- mommy: 0.2 − 1 = **−0.8** (the right answer gets probability − 1)
- bed: **0.1**

The pushes always add up to 0: what the right answer gains, the others lose.

</details>

In [ ]:
my_answer = None  # three numbers (milk, mommy, bed), e.g. [0.5, -0.5, 0]
check("3.3", my_answer)

### 3.4 A number is 0.82, its push is +0.2 and the learning rate is 0.1. What is the number after the step?

<details>
<summary>Show the answer</summary>
<!-- answer: 0.8 | 0.80 -->
<!-- format: one number -->

new = old − learning rate × push = 0.82 − 0.1 × 0.2 = 0.82 − 0.02 = **0.80**. A positive push means "growing raises the surprise", so the number goes down.

</details>

In [ ]:
my_answer = None  # one number
check("3.4", my_answer)

<!-- notebook-only -->

### 3.5 (code) Write `my_softmax(scores)` yourself, with `np.exp` and `.sum()`.

<details>
<summary>Show the answer</summary>

```python
def my_softmax(scores):
    exps = np.exp(scores)
    return exps / exps.sum()
```

Subtracting `scores.max()` first, as in section 3.2, gives the same result and avoids huge numbers when the scores are big.

</details>

<!-- /notebook-only -->

In [ ]:
def my_softmax(scores):
    # your code here
    return None


check("3.5", my_softmax)

### 3.6 Think: why does the loss stop at 0.512 instead of going down to 0?

<details>
<summary>Show the answer</summary>
<!-- choices: *Some guesses can't be sure: after `_I` both `_am` and `_want` happen, and 3 answers follow `_am` and `_want` | It needs more rounds: 10,000 rounds would reach 0 | The learning rate is too small | The random start was unlucky -->

The data itself is uncertain for a model that only sees the last token. After `_I`, both `_am` and `_want` happen. After `_am`, and after `_want`, three answers happen. The best it can do there is 1/2 and 1/3, which gives 0.512. More rounds, another learning rate or another start all end at the same floor.

</details>

### 3.7 Think: after training, the input rows of hungry, sad, tired, milk, mommy and bed point the same way. Why?

<details>
<summary>Show the answer</summary>
<!-- choices: *They are all followed by `_.`, so they must all give the same guess | They are all feelings | They share letters | We told the model they belong together -->

An input row is used to guess **what comes next**. These six tokens are all followed by `_.`, so the same guess must come out of all six rows, and training pushes them in the same direction. The output rows tell another story: they group the tokens by what comes **before** them.

</details>

## Summary

- Learning = **guess, measure the surprise, push every number a little, repeat**.
- The smallest toddler has two tables: score of a candidate = (input row of the token seen) · (output row of the candidate).
- **Softmax** turns scores into probabilities. The **loss** is −ln(probability of the right answer).
- The **push** (gradient) is probability − 1 on the right answer's score and probability on the others; it flows back to the rows (**backpropagation**). Each step: new = old − learning rate × push.
- From random numbers, 300 rounds bring the loss from 4.13 to 0.516, and the toddler finds the counts of Step 0. The floor (0.512) comes from the data, not from the training.
- Input rows learn what comes **after** a token, output rows what comes **before** it.
- Next: **Step 4, positions**, and the toddler trains again.